# TensionMagnetique — `tension_magnetique`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validation of the **magnetic tension** post-treatment field of the standalone
magnetic (induction) equation:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: `tension_magnetique`. Per the
TRUST validation-form layout the other `Postraitement_fields` cases live in
their own sibling folders.

## What it validates

$$\mathbf{F}_t = \frac{1}{\mu_0}(\mathbf{b}\cdot\nabla)\mathbf{b},\qquad
(\mathbf{F}_t)_i = \frac{1}{\mu_0}\sum_j b_j\,\frac{\partial b_i}{\partial x_j}$$

The per-element gradient $\nabla b$ is reconstructed by **Green-Gauss**
(the kernel has no element-field gradient factory on PolyMAC: `Discret_Thyd::grad_T`
is a stub on this discretisation and the generic `Gradient` post operator rejects
PolyMAC element fields), so the module computes it itself from the P0 unknown.

## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | magnetic tension force density | $F_t = \mu_0^{-1}(b\cdot\nabla)b$ |

Vector model (`nb_composantes 2`): the scalar model is rejected with a clean
error by `creer_champ` (`(b\cdot\nabla)b` is only meaningful for a full vector b).



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.



In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

print("BUILD =", BUILD)



## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).



In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "tension_magnetique.data", nbProcs=NB_PROCS)
case.partition()          # trust -partition tension_magnetique.data 2 -> PAR_tension_magnetique.data + DOM_000*.Zones
# --- mesh-convergence scan (shared ladder: L1 = cells x2, L2 = cells x4) ---
# Levels instantiated from src/tension_magnetique.tpl.data (byte-copy of the baseline
# dataset with $nx/$ny node-count placeholders); the baseline case above is
# the untouched L0. Each level runs in its own build/scan/<lvl>/ directory
# so that the partitioned DOM_000*.Zones never overwrite another's.
NX0 = NY0 = 11                       # baseline node counts (L0)
LEVELS = {
    "L1": {"nx": 2*11-1, "ny": 2*11-1},
    "L2": {"nx": 4*11-3, "ny": 4*11-3},
}
scan_cases = {}
for lvl, dic in LEVELS.items():
    c = run.addCaseFromTemplate("tension_magnetique.tpl.data", f"scan/{lvl}", dic=dic,
                                targetData=f"tension_magnetique_{lvl}.data",
                                nbProcs=NB_PROCS, excluNR=True)
    c.partition()
    scan_cases[lvl] = c
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)


## Checks


In [ ]:
entries = read_son_all(case.dataFileName_, "SONDE_FT")
for name, v in entries:
    print(name, "=", v[-1])
comps = [v[-1] for name, v in entries]

# b = (y, x) is exactly linear => grad b = [[0,1],[1,0]] at every cell centre,
# so Ft = (b . grad) b / mu0 = (x, y) with mu0 = 1.
expected = np.array([0.45, 0.55])
print(f"tension_magnetique (probe) : ({comps[0]:.6g}, {comps[1]:.6g})  (expected {expected})")
err = np.max(np.abs(np.array(comps) - expected)) / np.max(np.abs(expected))
print(f"relative error             : {err:.3e}")
assert err < 1e-12, f"tension_magnetique wrong: {comps}"
print("OK: tension_magnetique = (b . grad) b / mu0.")



## Mesh convergence

The shared refinement ladder: `L0` = the baseline mesh above (`11x11`),
`L1` = cells x2 (`21x21`), `L2` = cells x4 (`41x41`).

The field is reconstructed from a linear/uniform `b` at an interior probe,
which is a **discrete identity** (the Green-Gauss reconstruction is exact
for linear fields at interior cells): b = (y,x) linear => (b . grad) b / mu0 = (x, y), position-dependent. The scan asserts the same
machine-precision bound at **every** level
(evaluated at the probe's effective cell-centre position from the .son header) — no order is reported.

In [ ]:
def probe_pos(path):
    """Effective (x, y) of a point probe, from the .son header."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# Temps"):
                toks = line.replace("=", " = ").split()
                return (float(toks[toks.index("x") + 2]), float(toks[toks.index("y") + 2]))
    raise RuntimeError(f"no probe position header in {path}")
def read_final_all_dir(subdir, case_name, name):
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    return [np.asarray(son.getValues(e)[1], dtype=float) for e in son.getEntries()]

def level_error(subdir, case_name):
    xy = probe_pos(f"{BUILD}/{subdir}/{case_name}_SONDE_FT.son")
    comps = [float(vv[-1]) for vv in read_final_all_dir(subdir, case_name, "SONDE_FT")]
    expected = np.array([xy[0], xy[1]])
    e = float(np.max(np.abs(np.array(comps) - expected)))
    return e, expected

scan = {"L0": ("", case.dataFileName_),
        "L1": (f"scan/L1", scan_cases["L1"].dataFileName_),
        "L2": (f"scan/L2", scan_cases["L2"].dataFileName_)}
scan_nodes = {"L0": (11, 11), "L1": (21, 21), "L2": (41, 41)}
res = {lvl: level_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes        abs err        rel")
for lvl in ("L0", "L1", "L2"):
    n0, n1 = scan_nodes[lvl]
    e, expected = res[lvl]
    rel = e / float(np.max(np.abs(expected)))
    print(f"{lvl:<6} {n0:>3}x{n1:>3} {e:<13.3e} {rel:.1e}")
    assert rel < 1e-12, f"identity broken at {lvl}: {rel:.3e}"
print("OK: the reconstruction identity holds at every mesh level.")

## Computer Performance


In [ ]:
run.tablePerf()

